<img src="./src/logo.png" width="250">

**Baustein:** Zeitreihenanalyse  $\rightarrow$ **Subbaustein:** Zeitreihen und
CNNs $\rightarrow$ **Praktikum**

**Version:** 1.0, **Lizenz:** <a rel="license" href="http://creativecommons.org/licenses/by-nc-nd/4.0/">CC BY-NC-ND 4.0</a>

***

# Zeitreihenanalyse: Zeitreihen und CNNS

## Importieren der notwendigen Python-Bibliotheken

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from pandas import read_csv
import seaborn as sn

from scipy.signal import medfilt, sosfilt, butter

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import OneHotEncoder

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dropout, MaxPooling1D, Conv1D, Flatten, Dense, Input	
from tensorflow.compat.v1.logging import set_verbosity, ERROR

set_verbosity(ERROR) # unterdrücke Warnings von Tensorflow

---
## Festlegen wichtiger Variablen

In [ ]:
LABELS = ['Gehen', 'Treppe_hoch', 'Treppe_runter', 'Sitzen', 'Stehen', 'Liegen']
SIGNALS = ["acc_x", "acc_y", "acc_z"]

SAMPLING_RATE = 50  # Datenpunkte pro Sekunde / Hz
WINDOW_SIZE = 2.56  # Fenstergröße in Sekunden
OVERLAP = 0.5  # Überlappung
N_SAMPLES = int(WINDOW_SIZE * SAMPLING_RATE)  # Anzahl der Samples pro Fenster
STEP_SIZE = int(N_SAMPLES * (1 - OVERLAP))  # Schrittgröße für das Fenster

---
## Importieren der Daten aus den entsprechenden txt-files

In [ ]:
paths = ['./data/' + signal + '.txt' for signal in SIGNALS]
X_signals = []
for signal_type_path in paths:
    file = open(signal_type_path, 'r')
    X_signals.append(
        [np.array(serie, dtype=np.float32) for serie in [
            row.replace('  ', ' ').strip().split(' ') for row in file
        ]]
    )
    file.close()
X = np.transpose(np.array(X_signals), (1, 2, 0))
true_labels = np.loadtxt('./data/labels.txt',  dtype=np.int32).reshape(-1, 1)

---
### Aufgabe 1: Machen Sie sich mit dem Datensatz in ```X```, bzw. ```true_labels``` vertraut. Lesen Sie dafür auch die Beschreibung der Daten in der *README.txt*. Was sind die Dimensionen ```X```, bzw. ```true_labels``` von und wofür stehen die einzelnen Dimensionen? 

Antwort:

---
### Aufgabe 2: Sind die Klassen gleichmäßig verteilt oder handelt es sich hier um *imbalanced data*?

In [ ]:
_, counts = np.unique(true_labels, return_counts=True)
plt.figure(figsize=(8, 4))
plt.bar(LABELS, counts)
plt.ylabel('Anzahl')
plt.xlabel('Klasse')
plt.show()

Antwort:

---
### Aufgabe 3: Wieso sollte die Variable ```true_labels``` One-Hot-Encoded werden? Führen Sie das Encoding mithilfe dem ```One-Hot-Encoder``` von scikit-learn durch und überprüfen und erklären Sie die resultierende Dimension mit ```np.shape()```.

In [ ]:
# OneHotEncoder initialisieren
encoder = OneHotEncoder(sparse_output=False)
# Labels in kategoriale (One-Hot-kodierte) Form umwandeln
Y = 

Antwort:

---
## Split des Datensatz in Trainings- und Validierungsdaten

In [ ]:
# Initialize StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Splitting dataset into training and validation set
for train_index, val_index in sss.split(X, Y):
    X_train, X_val = X[train_index], X[val_index]
    y_train, y_val = Y[train_index], Y[val_index]

---
## Visualisieren der Zeitreihen für die einzelnen Klassen
### Aufgabe 4: Könnten Sie die Zeitreihen mit dem Auge klassifizieren? Woran machen Sie Ihre Klassifikationsentscheidung fest? Wäre eine Klassifikation mit nur einem Sensor in Ihren Augen möglich?

In [ ]:
idx_values = np.unique([tuple(row) for row in y_train], axis=0, return_index=True)[1].tolist() # Finden einer Zeitreihe für jede einzelne Klasse

time = np.linspace(0,2.56,128)

fig, axs = plt.subplots(2, 3, figsize=(10, 5))
for idx, ax in zip(idx_values, axs.flatten()):
    for i in range(len(SIGNALS)):
        ax.plot(time, X_train[idx, :, i], label=SIGNALS[i])

    ax.legend()
    ax.set_xlabel('Zeit in s')
    ax.set_ylabel('Beschleunigung in g')
    ax.set_title(LABELS[y_train[idx].argmax(axis = -1)])
plt.tight_layout()
plt.show()

Antwort:

---
## Erstellen, Kompilieren und Trainieren eines CNNs für die Klassifikation der Zeitreihen
### Aufgabe 5: Verändern Sie die unten gezeigte Netzarchitektur indem Sie z.B. Schichten einfügen, Kernel-Größen verändern, mehr Epochen laufen lassen usw. 

Machen Sie sich Notizen was das für Auswirkungen auf die Accuracy des Validierungs-Datensatz und die Confusion Matrix hat. Mit ```model.summary()``` können Sie sich die Netzwerkarchitektur übersichtlich ausgeben lassen. Fahren Sie dann mit der für Sie am besten funktionierenden Netzwerkarchitektur fort.

In [ ]:
verbose, epochs, batch_size = 1, 10, 32
n_timesteps, n_features, n_outputs = X_train.shape[1], X_train.shape[2], y_train.shape[1]

model = Sequential()
model.add(Input(shape=(n_timesteps,n_features)))
model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=4))
model.add(Flatten())
model.add(Dense(100, activation='relu'))
model.add(Dense(n_outputs, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy']) # Kompilieren des Netzes

model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=verbose, validation_data=(X_val, y_val)) # Trainieren des Netzes

In [ ]:
model.summary()
pred_y_val = model.predict(X_val)
ConfusionMatrixDisplay.from_predictions(y_val.argmax(axis=-1), pred_y_val.argmax(axis=-1), cmap='Blues')

Antwort/Notizen:

---
## Klassifikation eigener Daten
### Aufgabe 6: Nehmen Sie eine eigene Zeitreihe auf, die klassifiziert werden soll.


Nutzen Sie hierfür die App *phyphox* und legen Sie ein *Neues Experiment* an mit dem *Beschleunigungssensor*. 

WICHTIG: Vergleichen Sie mit der *README.txt*, welche **Sensorrate (Hz)** gewählt werden sollte. 

Da die **Ausrichtung** des Handys für die richtige Erfassung der Daten der Sensoren wichtig ist, halten Sie für die Messung Ihr Smartphone **waagerecht** auf der **rechten Hüfte/rechten Bauch**. Das Display zeigt von Ihnen weg und der obere Teil des Handys schaut nach rechts. (Schauen Sie sich, falls Sie wegen der Positionierung unsicher sind, das Video https://www.youtube.com/watch?v=XOEN9W05_4A an) 

**Starten** Sie eine Messung und führen Sie einige der **Bewegungen** aus, die klassifiziert werden sollen. 

Merken Sie sich in welcher Reihenfolge/wann ungefähr Sie was gemacht haben oder tun Sie sich in Zweiteams zusammen und lassen Sie abwechselnd die andere Person notieren, wann welche Bewegung gemacht wurde.

Pausieren Sie dann die Aufzeichnung und **exportieren** Sie die Daten unter den drei Punkten und *Daten exportieren* als csv-Datei *CSV (Comma, decimal point)*. Schicken Sie sich die Daten per Mail und speichern Sie die *Accelerometer.csv* in dem Ordner *data/HAR/*.

---
### Aufgabe 7: Importieren Sie Ihre Zeitreihe mithilfe der Pandas-Funktion ```read_csv()``` und speichern Sie die Zeitreihe in der  Variable ```data_acc``` ab.

---
### Aufgabe 8: Erklären Sie den nachfolgenden Code-Abschnitt für die Vorverarbeitung der Daten. 

Welche Vorverarbeitungsschritte werden hier angewandt und warum (vgl. Sie mit der *README.txt*)?
Nutzen Sie, falls Ihnen Funktionen nicht bekannt sind, die Dokumentation von ```scipy```.

In [ ]:
data_acc.drop('Time (s)', axis=1, inplace=True)

# Vorverarbeiten der Daten
for i in range(len(SIGNALS)):
    data_acc.iloc[:,i] = medfilt(data_acc.iloc[:,i])
    
    sos = butter(3, 20/(0.5*SAMPLING_RATE), 'low', output='sos')
    data_acc.iloc[:,i] = sosfilt(sos, data_acc.iloc[:,i]) # Butterworth-Filter
    
    data_acc.iloc[:,i] = data_acc.iloc[:,i]/9.80665 

display(data_acc)

Antwort:

---
### Aufgabe 9: Passen Sie ggf. die Reihenfolge der Sensoren an. 

Es kann sein, dass Ihr Smartphone andere Bezeichnungen der Sensorachsen hat, als das Smartphone aus den Trainingsdaten. Für die Klassifikation ist dies aber elementar. Vergleichen Sie hierfür z.B. Ihre Daten im Stehen und Liegen mit den der Trainingsdaten (Duplizieren Sie z.B. das Notebook um beide Abbildungen zu vergleichen) und passen Sie die Reihenfolge im folgenden Code an.

In [ ]:
time = np.linspace(0,len(data_acc)/SAMPLING_RATE,len(data_acc))

plt.figure()
for i in range(len(SIGNALS)):
    plt.plot(time,data_acc.iloc[:,i], label=SIGNALS[i])
plt.legend()
plt.xlabel('Zeit in s')
plt.ylabel('Beschleunigung in g')
plt.title('Eigener Testlauf')
plt.show()

In [ ]:
# Anpassen der Reihenfolge der Sensoren (Smartphone abhängig)
data_acc = data_acc[['Acceleration x (m/s^2)','Acceleration z (m/s^2)','Acceleration y (m/s^2)']]

---
## Teilen der Zeitreihe in einzelne Zeitfenster für die Klassifikation

Im Nachfolgenden wird die gesamte Zeitreihe (2D-Array) in viele einzelne Fenster der Länge 2.56s und mit 50% Überlappung aufgeteilt. Das resultierende Array in ```X_test``` ist demnach wieder ein 3D-Array. 

In [ ]:
# Anzahl der Fenster berechnen
total_samples = len(data_acc)
n_windows = (total_samples - N_SAMPLES) // STEP_SIZE + 1

# Initialisieren des 3D-Arrays
n_sensors = len(SIGNALS)  # Anzahl der Sensoren
X_test = np.empty((n_windows, N_SAMPLES, n_sensors))

# Daten in das 3D-Array füllen
for i in range(n_windows):
    start = i * STEP_SIZE
    end = start + N_SAMPLES
    X_test[i] = data_acc.iloc[start:end].values

---
## Klassifikation der eigenen Zeitreihen

Im Nachfolgenden werden die vielen Zeitabschnitte mithilfe des trainierten Netzes ```model``` klassifiziert. 

In [ ]:
pred_X_test = model.predict(X_test)
pred_labels_X_test = pred_X_test.argmax(axis = -1)

---
## Visualisierung der Klassifikation

Die nachfolgende Visualisierung zeigt die vorhergesagten Labels für die einzelnen Zeitabschnitte für den gesamten Zeitbereich. 

In [ ]:
colors = {0: 'red', 1: 'green', 2: 'blue', 3: 'cyan', 4: 'magenta', 5: 'yellow'}

time = np.linspace(0,len(data_acc)/SAMPLING_RATE,len(data_acc))
plt.figure()
for i in range(len(SIGNALS)):
    plt.plot(time,data_acc.iloc[:,i], label=SIGNALS[i])
ymin, ymax = plt.gca().get_ylim()    
for i, label in enumerate(pred_labels_X_test):
    # Berechne den Zeitpunkt für das Label (angepasst an die Daten)
    label_time = (i * 64 + 64 / 2) / 50  # Mitte des Labels
    if label_time < max(time):
        plt.scatter(label_time, ymin-1, color=colors[label], label=LABELS[label], s=100)

# Vermeidung von doppelten Legenden-Einträgen
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(),bbox_to_anchor=(1.04, 0.5), loc="center left")

plt.xlabel('Zeit in s')
plt.ylabel('Beschleunigung in g')
plt.title('Eigener Testlauf')
plt.show()

---
## Visualisierung einzelner Zeitfenster

### Aufgabe 10: Suchen Sie sich interessante Zeitfenster in der Zeitreihe aus, die einzeln visualisiert werden sollen.


Um auch die einzelnen Zeitfenster so wie bei den Trainingsdaten mit dem vorhergesagten Label anzeigen lassen zu können, können Sie die 6 interessante Zeitpunkte (z.B. 21s, 25s, 45s, ...) aus der obenstehenden Abbildung heraussuchen und in die Liste ```TOI``` eintragen. Die dazugehörigen Zeitfenster werden dann in der untenstehenden Abbildung visualisiert.

In [ ]:
TOI = []

In [ ]:
def find_index_of_timewindow(desired_timepoint, window_size=WINDOW_SIZE, overlap=OVERLAP):
    index = desired_timepoint // (window_size*(1-overlap))
    return int(index)

idx_TOI = [find_index_of_timewindow(timepoint) for timepoint in TOI]

time = np.linspace(0,WINDOW_SIZE,N_SAMPLES)

fig, axs = plt.subplots(2, 3, figsize=(10, 5))
for idx, ax in zip(idx_TOI, axs.flatten()):
    for i in range(len(SIGNALS)):
        ax.plot(time, X_test[idx, :, i], label=SIGNALS[i])

    ax.legend()
    ax.set_xlabel('Zeit in s')
    ax.set_ylabel('Beschleunigung in g')
    ax.set_title('Vorhersage:\n'+ LABELS[pred_labels_X_test[idx]])
plt.tight_layout()
plt.show()

---
### Aufgabe 11: Vergleichen Sie die Ergebnisse der Klassifikation Ihres Datensatzes mit der Accuracy des Validierungs-Datensatz. Sind die Ergebnisse vergleichbar? Warum (nicht)?

Antwort:

---

<a rel="license" href="http://creativecommons.org/licenses/by-nc-nd/4.0/"><img alt="Creative Commons Lizenzvertrag" style="border-width:0" src="https://i.creativecommons.org/l/by-nc-nd/4.0/88x31.png" /></a><br /><span xmlns:dct="http://purl.org/dc/terms/" property="dct:title">Die Übungsserie begleitend zum AI4ALL-Kurs</span> der <span xmlns:cc="http://creativecommons.org/ns#" property="cc:attributionName">EAH Jena</span> ist lizenziert unter einer <a rel="license" href="http://creativecommons.org/licenses/by-nc-nd/4.0/">Creative Commons Namensnennung - Nicht kommerziell - Keine Bearbeitungen 4.0 International Lizenz</a>.

Der AI4ALL-Kurs entsteht im Rahmen des Projekts MoVeKI2EAH. Das Projekt MoVeKI2EAH wird durch das BMBF (Bundesministerium für Bildung und Forschung) und den Freistaat Thüringen im Rahmen der Bund-Länder-Initiative zur Förderung von Künstlicher Intelligenz in der Hochschulbildung gefördert (12/2021 bis 11/2025, Föderkennzeichen 16DHBKI081).